Additional Features:
 - Excess style momentum
 - Excess style volatility (against benchmark)
 - Excess style kurtosis (against benchmark)
 - Excess style skewness (against benchmark)
 - Relative momentum (factors relative to each other, trend scan - categorical features)
 - Short-technical indicators (remove)
 - Medium technical indicators (remove)
 - Omega ration (remove)


In [30]:
# run script from the data_input file
import os
os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')

In [31]:
# Import libraries:
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew


In [32]:
# Import data:
df_benchmark = pd.read_csv("benchmark_returns.csv", parse_dates=['Date'])
df_benchmark = df_benchmark.set_index('Date')

df_factors = pd.read_csv("factor_returns.csv", parse_dates=['Date'])
df_factors = df_factors.set_index('Date')

df = pd.concat([df_benchmark, df_factors], axis=1, join='inner')


print(df)
print(df.dtypes)


            Benchmark Momentum  Value  Quality
Date                                          
2004-02-01       0.04    0.02    0.01      NaN
2004-03-01      -0.00    0.01    0.01      NaN
2004-04-01       0.00   -0.02    0.01      NaN
2004-05-01      -0.02    0.00    0.00      NaN
2004-06-01       0.01    0.00    0.01      NaN
...               ...      ...    ...      ...
2023-10-01      -0.03   -0.03   -0.06    -0.03
2023-11-01      -0.03    0.10    0.06     0.09
2023-12-01       0.08    0.01    0.04     0.02
2024-01-01       0.03   -0.02   -0.05    -0.03
2024-02-01      -0.03   -0.02   -0.05    -0.01

[241 rows x 4 columns]
Benchmark    float64
Momentum      object
Value        float64
Quality      float64
dtype: object


In [33]:
cols = ['Benchmark', 'Momentum', 'Value', 'Quality']

for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')  

print(df)
print(df.dtypes)

            Benchmark  Momentum  Value  Quality
Date                                           
2004-02-01       0.04      0.02   0.01      NaN
2004-03-01      -0.00      0.01   0.01      NaN
2004-04-01       0.00     -0.02   0.01      NaN
2004-05-01      -0.02      0.00   0.00      NaN
2004-06-01       0.01      0.00   0.01      NaN
...               ...       ...    ...      ...
2023-10-01      -0.03     -0.03  -0.06    -0.03
2023-11-01      -0.03      0.10   0.06     0.09
2023-12-01       0.08      0.01   0.04     0.02
2024-01-01       0.03     -0.02  -0.05    -0.03
2024-02-01      -0.03     -0.02  -0.05    -0.01

[241 rows x 4 columns]
Benchmark    float64
Momentum     float64
Value        float64
Quality      float64
dtype: object


In [34]:
# Excess returns:

styles = ['Momentum','Value','Quality']
for style in styles:
    df[f'{style}_excess'] = df[style] - df['Benchmark']

In [35]:
# Excess momentum, volatility, kurtosis and skewness:

window = 6

for style in styles:
    # Excess Style Momentum (Cumulative excess return)
    df[f'{style}_excess_momentum'] = df[f'{style}_excess'].rolling(window).sum()
    

    # Excess Style Volatility
    df[f'{style}_excess_volatility'] = df[f'{style}_excess'].rolling(window).std()
    

    # Excess Style Kurtosis
    df[f'{style}_excess_kurtosis'] = df[f'{style}_excess'].rolling(window).apply(kurtosis, raw=True)

    # Excess Style Skewness
    df[f'{style}_excess_skewness'] = df[f'{style}_excess'].rolling(window).apply(skew, raw=True)
    
    


In [36]:
# Compare factors risk to benchmark risk:

# Rolling risk stats for benchmark
df['benchmark_vol'] = df['Benchmark'].rolling(window).std()
df['benchmark_kurtosis'] = df['Benchmark'].rolling(window).apply(kurtosis, raw=True)
df['benchmark_skewness'] = df['Benchmark'].rolling(window).apply(skew, raw=True)

for style in styles:
    df[f'{style}_relative_volatility'] = df[f'{style}'].rolling(window).std() / df['benchmark_vol']
    df[f'{style}_relative_kurtosis'] = df[f'{style}'].rolling(window).apply(kurtosis, raw=True) / df['benchmark_kurtosis']
    df[f'{style}_relative_skewness'] = df[f'{style}'].rolling(window).apply(skew, raw=True) / df['benchmark_skewness']


In [37]:
# Technical indicators:
for style in styles:
    df[f'{style}_momentum_1M'] = df[style].shift(1)
    df[f'{style}_momentum_3M'] = df[style].rolling(3).sum()
    df[f'{style}_momentum_6M'] = df[style].rolling(6).sum()
    df[f'{style}_momentum_12M'] = df[style].rolling(12).sum()
    df[f'{style}_momentum_24M'] = df[style].rolling(24).sum()


In [43]:
# Relative factor performance:

df = df.drop(['Benchmark','Momentum','Value','Quality','Momentum_excess','Value_excess','Quality_excess',
              'benchmark_vol','benchmark_kurtosis','benchmark_skewness'],axis=1)
print(df.columns)


Index(['Momentum_excess_momentum', 'Momentum_excess_volatility',
       'Momentum_excess_kurtosis', 'Momentum_excess_skewness',
       'Value_excess_momentum', 'Value_excess_volatility',
       'Value_excess_kurtosis', 'Value_excess_skewness',
       'Quality_excess_momentum', 'Quality_excess_volatility',
       'Quality_excess_kurtosis', 'Quality_excess_skewness',
       'Momentum_relative_volatility', 'Momentum_relative_kurtosis',
       'Momentum_relative_skewness', 'Value_relative_volatility',
       'Value_relative_kurtosis', 'Value_relative_skewness',
       'Quality_relative_volatility', 'Quality_relative_kurtosis',
       'Quality_relative_skewness', 'Momentum_momentum_1M',
       'Momentum_momentum_3M', 'Momentum_momentum_6M', 'Momentum_momentum_12M',
       'Momentum_momentum_24M', 'Value_momentum_1M', 'Value_momentum_3M',
       'Value_momentum_6M', 'Value_momentum_12M', 'Value_momentum_24M',
       'Quality_momentum_1M', 'Quality_momentum_3M', 'Quality_momentum_6M',
       

In [44]:
# Save output into csv file:
df.to_csv('additional_features.csv', index=True)

In [45]:
print(df.columns)

Index(['Momentum_excess_momentum', 'Momentum_excess_volatility',
       'Momentum_excess_kurtosis', 'Momentum_excess_skewness',
       'Value_excess_momentum', 'Value_excess_volatility',
       'Value_excess_kurtosis', 'Value_excess_skewness',
       'Quality_excess_momentum', 'Quality_excess_volatility',
       'Quality_excess_kurtosis', 'Quality_excess_skewness',
       'Momentum_relative_volatility', 'Momentum_relative_kurtosis',
       'Momentum_relative_skewness', 'Value_relative_volatility',
       'Value_relative_kurtosis', 'Value_relative_skewness',
       'Quality_relative_volatility', 'Quality_relative_kurtosis',
       'Quality_relative_skewness', 'Momentum_momentum_1M',
       'Momentum_momentum_3M', 'Momentum_momentum_6M', 'Momentum_momentum_12M',
       'Momentum_momentum_24M', 'Value_momentum_1M', 'Value_momentum_3M',
       'Value_momentum_6M', 'Value_momentum_12M', 'Value_momentum_24M',
       'Quality_momentum_1M', 'Quality_momentum_3M', 'Quality_momentum_6M',
       